# Capacity-matched graph ablations — is it the content or the parameter count?

Every arm that removes edge types from the KG also shrinks the encoder, because `HeteroRGCNLayer`
builds **one `Linear(n_inp, n_hid)` plus one `Linear(n_hid, n_out)` per edge type**. Keeping 9 of 50
edge types therefore drops the encoder from 1,010,000 to 181,800 parameters — 82% gone — so the
measured loss mixes *less graph* with *less model*.

Grid 6 closed that confound for one arm (`only_disease_matched`: 9 edge types widened to
`n_hid = n_out = 288`, 1,010,880 parameters, and 40% of the reported loss turned out to be capacity).
**This notebook extends that recipe to every remaining reduced-capacity graph arm**, so that every
keep-only number in the paper can be quoted at matched capacity rather than as an upper bound.

| arm | edge types | width | encoder params | status |
|---|---|---|---|---|
| `full_kg` | 50 | 100 | 1,010,000 | imported |
| `only_disease_relations` | 9 | 100 | 181,800 | imported |
| `only_disease_matched` | 9 | 288 | 1,010,880 | imported (grid 6) |
| `dd_plus_padding` | 9 | 100 | 181,800 | imported |
| `dp_plus_padding` | 9 | 100 | 181,800 | imported |
| **`dd_plus_padding_matched`** | 9 | 288 | 1,010,880 | **new** |
| **`dp_plus_padding_matched`** | 9 | 288 | 1,010,880 | **new** |
| `dd_only` | 6 | 100 | 121,200 | imported |
| **`dd_only_matched`** | 6 | 362 | 1,007,808 | **new** |

### How the widening works, and what it does not control

`n_hid` and `n_out` are raised together; **`n_inp` stays at 100**. Widening the input too would also
enrich the frozen random input features — a second, uncontrolled change — and would overshoot the
parameter target. This is exactly the choice grid 6 made, so the new arms are directly comparable to
`only_disease_matched`.

Widening `n_out` also widens the decoder: `w_rels` becomes `(n_etypes, n_out)` and the `rarity` gate
becomes `Linear(n_out·2, 1)`. Those add ~3k parameters against the encoder's ~1.01M, and the change is
identical in `only_disease_matched`, so it does not differentiate the arms. `n_params` is recorded
alongside `encoder_params` so both can be reported.

### What each new arm answers

- **`dd_plus_padding_matched` / `dp_plus_padding_matched`** — grid 6's Result B (identity beats count,
  `dp − dd` = +0.0629 indication) was measured at 181,800 parameters. These re-measure it at 1.01M. If
  the gap holds, relation *identity* matters at full capacity too; if it shrinks, part of it was the
  narrow encoder interacting with content.
- **`dd_only_matched`** — `dd_only` scores exactly 0.5000 ± 0.0000 on every seed, because dropping
  `disease_disease` and `disease_protein` empties the disease signature. The prediction is that **a 8.3×
  larger encoder changes nothing at all**: the arm is information-starved, not capacity-starved. A
  confirmed 0.5000 here is the cleanest possible proof that the degeneracy is not a capacity artifact,
  which is the one objection grid 5 could not answer.

Because `dd_only` is predicted degenerate, it runs on a short seed list by default
(`DEGENERATE_SEEDS`); widen it only if a reviewer asks.

### Reading the results

The quantity of interest is the **capacity share** of each previously reported loss:

```
capacity share = 1 − (matched delta / narrow delta)
```

Grid 6 found 40% for `only_disease_relations`. Section 8 computes it for every pair.

---
## 1 · Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'
sys.path.insert(0, str(PYG_ROOT))   # PyG port FIRST, so `import txgnn` never hits dgl_implementation/

import torch
import torch.nn as nn

import txgnn
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit this notebook.')

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print('repo  :', REPO_ROOT)
print('txgnn :', txgnn.__file__)
print('torch :', torch.__version__, '| device:', DEVICE,
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())

---
## 2 · Configuration

Seeds, split and training schedule are identical to grids 5 and 6, so every delta is paired per seed.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
PROFILE = 'full'

MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

TARGET_ENCODER_PARAMS = 1_010_000   # the full 50-edge-type encoder at width 100

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

SIG_ETYPES = ['disease_disease', 'rev_disease_protein']   # library default, model.py:88

DATA_FOLDER  = REPO_ROOT / 'data'
ABL_ROOT     = REPO_ROOT / 'results' / 'ablation_capacity_matched_graph'
KEEP_ROOT    = REPO_ROOT / 'results' / 'ablation_keep_disease_only'    # read-only
CONTROL_ROOT = REPO_ROOT / 'results' / 'ablation_keep_only_controls'   # read-only
ABL_ROOT.mkdir(parents=True, exist_ok=True)

REF_ARM   = 'full_kg'                  # primary reference for every delta
REF_ARM_2 = 'only_disease_relations'   # secondary reference

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True
IMPORT_REFS       = True    # pull finished arms from grids 5 and 6 instead of retraining them
DEGENERATE_SEEDS  = [1, 2, 3]   # dd_only_matched is predicted to be exactly 0.500
STOP_AFTER_N_RUNS = None

print('split  :', SPLIT)
print('seeds  : %d -> %s' % (len(SEEDS), SEEDS))
print('results:', ABL_ROOT.relative_to(REPO_ROOT))

---
## 3 · Arms

`matched_width` solves for the width that lands closest to 1,010,000 encoder parameters for a given
edge-type count. Encoder parameters for `n` edge types at `n_inp = 100`, `n_hid = n_out = H` are

$$n \times \big[(100H + H) + (H^2 + H)\big] \;=\; n\,(H^2 + 102H)$$

The solved widths are asserted against the model that actually gets built, on every run.

In [ ]:
DD_RELATIONS        = ['indication', 'contraindication', 'off-label use']
SIGNATURE_RELATIONS = ['disease_disease', 'disease_protein']

# inert padding, carried over from grid 6 unchanged so the arms stay comparable
PAD_FOR_DP = 'molfunc_protein'      # 2 etypes, 139,060 edges  <- stands in for disease_protein
PAD_FOR_DD = 'phenotype_phenotype'  # 1 etype,   37,472 edges  <- stands in for disease_disease

SAME_TYPE_RELATIONS = {
    'disease_disease', 'drug_drug', 'protein_protein', 'phenotype_phenotype', 'anatomy_anatomy',
    'molfunc_molfunc', 'cellcomp_cellcomp', 'pathway_pathway', 'bioprocess_bioprocess',
    'exposure_exposure',
}


def n_etypes_for(keep):
    '''Edge types a `keep` list produces: 1 for same-type relations, 2 otherwise.'''
    return sum(1 if r in SAME_TYPE_RELATIONS else 2 for r in keep)


def expected_encoder_params(n_etypes, n_inp, n_hid, n_out):
    return n_etypes * ((n_inp * n_hid + n_hid) + (n_hid * n_out + n_out))


def matched_width(n_etypes, target=TARGET_ENCODER_PARAMS, n_inp=100, hi=2048):
    '''Width H (= n_hid = n_out) whose encoder params land closest to `target`.'''
    best = None
    for H in range(8, hi + 1):
        p = expected_encoder_params(n_etypes, n_inp, H, H)
        if best is None or abs(p - target) < abs(best[1] - target):
            best = (H, p)
    return best


W9, P9 = matched_width(9)
W6, P6 = matched_width(6)

# grid 6 solved the 9-edge-type case by hand; this must reproduce it or the arms are not comparable
assert (W9, P9) == (288, 1_010_880), 'expected grid 6 width 288 / 1,010,880, got %s / %s' % (W9, P9)

print('9 edge types -> n_hid = n_out = %d  ->  %s encoder params (%+.2f%% of target)'
      % (W9, format(P9, ','), 100.0 * (P9 - TARGET_ENCODER_PARAMS) / TARGET_ENCODER_PARAMS))
print('6 edge types -> n_hid = n_out = %d  ->  %s encoder params (%+.2f%% of target)'
      % (W6, format(P6, ','), 100.0 * (P6 - TARGET_ENCODER_PARAMS) / TARGET_ENCODER_PARAMS))

In [ ]:
def _arm(name, keep, overrides=None, note='', seeds=None, source=None, degenerate=False):
    cfg = dict(MODEL_CFG, **(overrides or {}))
    n_et = 50 if keep is None else n_etypes_for(keep)
    return {'arm': name, 'keep': keep, 'overrides': overrides or {}, 'note': note,
            'expect_etypes': n_et, 'seeds': seeds, 'source': source, 'degenerate': degenerate,
            'expect_encoder_params': expected_encoder_params(
                n_et, cfg['n_inp'], cfg['n_hid'], cfg['n_out'])}


WIDE9 = {'n_hid': W9, 'n_out': W9}
WIDE6 = {'n_hid': W6, 'n_out': W6}

KEEP_DD_ONLY   = DD_RELATIONS
KEEP_DISEASE_9 = DD_RELATIONS + SIGNATURE_RELATIONS
KEEP_DD_PAD    = DD_RELATIONS + ['disease_disease', PAD_FOR_DP]
KEEP_DP_PAD    = DD_RELATIONS + ['disease_protein', PAD_FOR_DD]

ARMS = [
    # ---- imported references, narrow ----
    _arm('full_kg', None, source=KEEP_ROOT,
         note='reference: unmodified KG'),
    _arm('only_disease_relations', KEEP_DISEASE_9, source=KEEP_ROOT,
         note='grid 5: 9 edge types, narrow encoder'),
    _arm('dd_only', KEEP_DD_ONLY, source=KEEP_ROOT, degenerate=True,
         note='grid 5: task edges only - empty signature, predicted 0.500'),
    _arm('dd_plus_padding', KEEP_DD_PAD, source=CONTROL_ROOT,
         note='grid 6: disease_disease only, padded to 9 edge types'),
    _arm('dp_plus_padding', KEEP_DP_PAD, source=CONTROL_ROOT,
         note='grid 6: disease_protein only, padded to 9 edge types'),

    # ---- imported reference, already matched ----
    _arm('only_disease_matched', KEEP_DISEASE_9, overrides=WIDE9, source=CONTROL_ROOT,
         note='grid 6: 9 edge types at matched capacity'),

    # ---- new ----
    _arm('dd_plus_padding_matched', KEEP_DD_PAD, overrides=WIDE9,
         note='NEW: identity-vs-count at matched capacity (disease_disease)'),
    _arm('dp_plus_padding_matched', KEEP_DP_PAD, overrides=WIDE9,
         note='NEW: identity-vs-count at matched capacity (disease_protein)'),
    _arm('dd_only_matched', KEEP_DD_ONLY, overrides=WIDE6, seeds=DEGENERATE_SEEDS, degenerate=True,
         note='NEW: predicted still exactly 0.500 - capacity cannot rescue an empty signature'),
]

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'

IMPORTABLE = {a['arm'] for a in ARMS if a['source'] is not None}
NEW_ARMS   = [a['arm'] for a in ARMS if a['source'] is None]


def arm_seeds(a):
    return a['seeds'] or SEEDS


display(pd.DataFrame([{
    'arm': a['arm'],
    'edge types': a['expect_etypes'],
    'width': dict(MODEL_CFG, **a['overrides'])['n_hid'],
    'encoder params': format(a['expect_encoder_params'], ','),
    '% of full': '%.1f%%' % (100.0 * a['expect_encoder_params'] / TARGET_ENCODER_PARAMS),
    'seeds': len(arm_seeds(a)),
    'source': 'new' if a['source'] is None else Path(a['source']).name,
} for a in ARMS]))

print('\nnew runs to train: %d arm(s) -> %s' % (len(NEW_ARMS), NEW_ARMS))
print('total new runs    : %d' % sum(len(arm_seeds(ARMS_BY_NAME[n])) for n in NEW_ARMS))

---
## 4 · Data and helpers

Identical to grids 5 and 6: one seed's frames held at a time, graph rebuilt per arm from the filtered train frame while `df` stays whole so node counts never move.

In [ ]:
def expand_relations(names):
    '''Relation base names -> the names present in the frames, incl. rev_ twins.'''
    out = set()
    for r in names:
        out |= {r, 'rev_' + r}
    return out


def keep_relations(df, keep):
    if keep is None:
        return df.reset_index(drop=True)
    return df[df.relation.isin(expand_relations(keep))].reset_index(drop=True)


class KeptData:
    '''Minimal TxData stand-in holding the reduced splits and a rebuilt graph.'''

    def __init__(self, base, keep, data_folder, split):
        self.df = base['df']                      # untouched: fixes node counts across arms
        self.df_train = keep_relations(base['df_train'], keep)
        self.df_valid = keep_relations(base['df_valid'], keep)
        self.df_test  = keep_relations(base['df_test'],  keep)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def signature_probe(model):
    '''(dim, max) of the disease signatures actually built. max == 0 => every signature is zero.'''
    try:
        prof = model.pred.diseases_profile_etypes
        et = next(iter(prof))
        vals = list(prof[et].values())[:64]
        if not vals:
            return None, None
        return int(vals[0].shape[0]), float(max(float(v.max()) for v in vals))
    except Exception:
        return None, None


_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the unablated split frames for one seed. Holds one seed at a time.'''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()

---
## 5 · Importing the finished arms

Six of the nine arms already exist in grids 5 and 6 at exactly this configuration. They are copied in
rather than retrained, but **only after every field that could make them a different experiment is
checked** — split, seed, edge-type count, the full model config including the width, and both training
schedules. A mismatch prints the reason and the arm is trained instead. Every copy is stamped with
`imported_from` so no imported run is ever counted as new compute.

In [ ]:
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import(arm_name, seed):
    if not IMPORT_REFS or arm_name not in IMPORTABLE:
        return False, 'not importable'
    a = ARMS_BY_NAME[arm_name]
    cfg = dict(MODEL_CFG, **a['overrides'])
    expect = {'split': SPLIT, 'seed': seed,
              'n_edge_types_after': a['expect_etypes'],
              'model_config': {k: cfg[k] for k in MODEL_KEYS_CHECKED},
              'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG}
    return import_matching_run(
        a['source'] / ('%s_seed%d' % (arm_name, seed)), run_dir(arm_name, seed), expect,
        extra={'experiment': 'capacity_matched_graph',
               'kept_relations_requested': a['keep'],
               'expect_encoder_params': a['expect_encoder_params']})


if IMPORT_REFS:
    for _name in sorted(IMPORTABLE):
        n_ok = n_fail = 0
        for s in arm_seeds(ARMS_BY_NAME[_name]):
            if is_done(_name, s):
                n_ok += 1
                continue
            ok, why = try_import(_name, s)
            if ok:
                n_ok += 1
            else:
                n_fail += 1
                if n_fail <= 2:
                    print('[import] %s_seed%d: will train (%s)' % (_name, s, why))
        print('[import] %-24s %2d present, %2d to train' % (_name, n_ok, n_fail))
else:
    print('IMPORT_REFS is False - every arm will be trained here.')

---
## 6 · The runner

Guards, all of them assertions that stop the run rather than warnings:

| guard | catches |
|---|---|
| `n_edge_types_after == expect_etypes` | a graph filter that kept the wrong relations |
| `encoder_params == expect_encoder_params`, exact integers | a "matched capacity" arm that is not matched |
| node-type set unchanged | a filter that silently dropped a node type |
| `signature_sources_present` | a signature that is present or empty when it should not be |
| `signature_max == 0` on the degenerate arms | the empty-signature mechanism not firing |

In [ ]:
def run_one(arm_name, seed, verbose=True):
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    ok, why = try_import(arm_name, seed)
    if ok:
        print('[skip] %s_seed%d -> %s' % (arm_name, seed, why))
        return pd.read_csv(el_csv)

    cfg = dict(MODEL_CFG, **arm['overrides'])

    print('\n' + '#' * 78)
    print('# %s   seed %d' % (arm_name, seed))
    print('# keep  = %s' % (arm['keep'],))
    print('# width %d/%d/%d | expect %d edge types, %s encoder params'
          % (cfg['n_inp'], cfg['n_hid'], cfg['n_out'], arm['expect_etypes'],
             format(arm['expect_encoder_params'], ',')))
    print('#' * 78)

    base = get_seed_data(seed)

    set_all_seeds(seed)
    d = KeptData(base, arm['keep'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    n_et = len(d.G.edge_types)
    rel_names = {et[1] for et in d.G.edge_types}

    # ---- guard: edge-type count ----
    assert n_et == arm['expect_etypes'], \
        '%s: %d edge types, expected %d -> %s' % (arm_name, n_et, arm['expect_etypes'], etypes_after)

    # ---- guard: node types untouched ----
    want_nodes = set(d.df.x_type) | set(d.df.y_type)
    assert set(d.G.node_types) == want_nodes, \
        'node types changed: %s' % (set(d.G.node_types) ^ want_nodes)

    sig_present = [r for r in SIG_ETYPES if r in rel_names]
    print('[graph] train edges %d of %d (%.2f%%) | %d edge types | signature sources %s'
          % (len(d.df_train), len(base['df_train']),
             100.0 * len(d.df_train) / len(base['df_train']), n_et, sig_present or 'NONE'))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**cfg)

    # ---- guard: capacity is exactly what the arm claims ----
    enc_p, tot_p = encoder_params(m.model), get_n_params(m.model)
    assert enc_p == arm['expect_encoder_params'], \
        '%s: encoder has %d params, expected %d' % (arm_name, enc_p, arm['expect_encoder_params'])

    sig_dim, sig_max = signature_probe(m.model)
    print('[model] encoder params %s (%.1f%% of full) | total %s | signature dim %s max %s'
          % (format(enc_p, ','), 100.0 * enc_p / TARGET_ENCODER_PARAMS,
             format(tot_p, ','), sig_dim, sig_max))

    # ---- guard: the degenerate arms must really have an empty signature ----
    if arm['degenerate']:
        assert not sig_present, '%s should have no signature source, has %s' % (arm_name, sig_present)
        if sig_max is not None:
            assert sig_max == 0.0, '%s signature is not empty (max %s)' % (arm_name, sig_max)
    else:
        assert sig_present, '%s lost its signature sources' % arm_name

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'experiment': 'capacity_matched_graph',
        'run_key': rdir.name, 'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'note': arm['note'],
        'kept_relations_requested': arm['keep'],
        'n_edge_types_after': n_et, 'edge_types_after': etypes_after,
        'n_node_types': len(d.G.node_types),
        'train_edges_after': int(len(d.df_train)),
        'train_edges_before': int(len(base['df_train'])),
        'model_config': cfg,
        'expect_encoder_params': arm['expect_encoder_params'],
        'encoder_params': int(enc_p), 'n_params': int(tot_p),
        'signature_sources_present': sig_present,
        'signature_dim': sig_dim, 'signature_max': sig_max,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': round(pre_s, 1), 'finetune_seconds': round(fin_s, 1),
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    }
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d  indication %.4f  contraindication %.4f  micro %.4f  (%.1f min)'
          % (arm_name, seed, ind, con, micro_auprc, (pre_s + fin_s) / 60))

    del m, d
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

---
## 7 · Run the grid

Resumable: a finished run is skipped on its files, so an interrupted grid picks up where it stopped.

In [ ]:
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS
            if s in arm_seeds(a) and not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending.' % len(todo))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 8 · Results

In [ ]:
def _load_all(root):
    recs = []
    for meta_p in sorted(Path(root).glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'encoder_params': meta.get('encoder_params'),
                         'n_params': meta.get('n_params'),
                         'n_edge_types': meta.get('n_edge_types_after'),
                         'train_edges': meta.get('train_edges_after'),
                         'width': (meta.get('model_config') or {}).get('n_hid'),
                         'sig_sources': ','.join(meta.get('signature_sources_present') or []),
                         'sig_max': meta.get('signature_max'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    return 1.96 * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all(ABL_ROOT)
wide = summary = None

if long.empty:
    print('No runs on disk yet - run section 7 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    _first = lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan
    info = long.groupby('arm').agg(encoder_params=('encoder_params', _first),
                                   n_params=('n_params', _first),
                                   n_edge_types=('n_edge_types', _first),
                                   width=('width', _first),
                                   train_edges=('train_edges', _first),
                                   sig_sources=('sig_sources', _first))

    rows = []
    ref_w = wide.xs(REF_ARM, level='arm') if REF_ARM in wide.index.get_level_values('arm') else None
    for a in wide.index.get_level_values('arm').unique():
        w = wide.xs(a, level='arm')
        r = {'arm': a, 'n_seeds': len(w)}
        for k in ROW:
            r[k] = w[k].mean()
            r['%s_ci' % k] = _ci95(w[k])
        if ref_w is not None and a != REF_ARM:
            common = w.index.intersection(ref_w.index)
            for k in ROW:
                dl = (w.loc[common, k] - ref_w.loc[common, k]).dropna()
                r['%s_delta' % k] = dl.mean()
                r['%s_delta_ci' % k] = _ci95(dl)
                r['%s_hurt' % k] = int((dl < 0).sum())
        rows.append(r)
    summary = (pd.DataFrame(rows).merge(info.reset_index(), on='arm', how='left')
               .sort_values('indication', ascending=False))

    show = summary[['arm', 'n_seeds', 'n_edge_types', 'width', 'encoder_params',
                    'indication', 'indication_ci', 'contraindication', 'contraindication_ci',
                    'micro', 'micro_ci']]
    print('Absolute AUPRC (chance = 0.500 on this protocol)')
    display(show.style.format({'indication': '{:.4f}', 'indication_ci': '±{:.4f}',
                               'contraindication': '{:.4f}', 'contraindication_ci': '±{:.4f}',
                               'micro': '{:.4f}', 'micro_ci': '±{:.4f}',
                               'encoder_params': '{:,.0f}'}, na_rep='-'))

### 8.1 The capacity share of each reported loss

For each narrow/matched pair, how much of the previously reported loss was the smaller encoder rather
than the missing graph content. Grid 6 found **40%** for `only_disease_relations`; that row is
reproduced here from the imported runs as a consistency check on this notebook's bookkeeping.

In [ ]:
PAIRS = [
    ('only_disease_relations',  'only_disease_matched',      '9 etypes, both signature relations'),
    ('dd_plus_padding',         'dd_plus_padding_matched',   '9 etypes, disease_disease only'),
    ('dp_plus_padding',         'dp_plus_padding_matched',   '9 etypes, disease_protein only'),
    ('dd_only',                 'dd_only_matched',           '6 etypes, no signature (degenerate)'),
]

if summary is None:
    print('Nothing to summarise yet.')
else:
    s = summary.set_index('arm')
    rows = []
    for narrow, matched, label in PAIRS:
        if narrow not in s.index or matched not in s.index:
            continue
        for key in ['indication', 'contraindication', 'micro']:
            dn = s.loc[narrow, '%s_delta' % key]
            dm = s.loc[matched, '%s_delta' % key]
            share = (1 - dm / dn) if (pd.notna(dn) and dn) else np.nan
            rows.append({'contrast': label, 'metric': key,
                         'narrow delta': dn, 'matched delta': dm,
                         'recovered': dm - dn,
                         'capacity share': share})
    if rows:
        t = pd.DataFrame(rows)
        display(t.style.format({'narrow delta': '{:+.4f}', 'matched delta': '{:+.4f}',
                                'recovered': '{:+.4f}', 'capacity share': '{:.0%}'}, na_rep='-'))
        print('\n"matched delta" is the number to quote. "narrow delta" is an upper bound on the loss,')
        print('because it also pays for an encoder 5.6x smaller.')
    else:
        print('No complete narrow/matched pair on disk yet.')

### 8.2 Identity versus count, re-measured at matched capacity

Grid 6's Result B: at 181,800 parameters and an identical 9 edge types, keeping `disease_protein`
beat keeping `disease_disease` by **+0.0629 ± 0.0284** indication and **+0.0845 ± 0.0368**
contraindication. The same contrast at 1,010,880 parameters is below. A gap that survives says the
result is about *which* relation, not about how much encoder the relation came with.

In [ ]:
def paired_contrast(arm_a, arm_b, label=None):
    '''Per-seed (arm_a - arm_b) with a t-CI and a sign count. Returns a dict or None.'''
    if wide is None:
        return None
    present = wide.index.get_level_values('arm').unique()
    if arm_a not in present or arm_b not in present:
        return None
    wa, wb = wide.xs(arm_a, level='arm'), wide.xs(arm_b, level='arm')
    common = wa.index.intersection(wb.index)
    if len(common) == 0:
        return None
    out = {'contrast': label or ('%s - %s' % (arm_a, arm_b)), 'n': len(common)}
    for k in ROW:
        d = (wa.loc[common, k] - wb.loc[common, k]).dropna()
        out[k] = d.mean()
        out['%s_ci' % k] = _ci95(d)
        out['%s_favours_a' % k] = int((d > 0).sum())
    return out


CONTRASTS = [
    ('dp_plus_padding',         'dd_plus_padding',         'dp - dd  @ 181,800 (grid 6)'),
    ('dp_plus_padding_matched', 'dd_plus_padding_matched', 'dp - dd  @ 1,010,880 (new)'),
    ('only_disease_matched',    'dp_plus_padding_matched', 'both relations - dp only @ matched'),
    ('only_disease_matched',    'dd_plus_padding_matched', 'both relations - dd only @ matched'),
]

rows = [r for r in (paired_contrast(a, b, lbl) for a, b, lbl in CONTRASTS) if r]
if rows:
    t = pd.DataFrame(rows)[['contrast', 'n', 'indication', 'indication_ci',
                            'indication_favours_a', 'contraindication', 'contraindication_ci',
                            'contraindication_favours_a', 'micro', 'micro_ci']]
    display(t.style.format({'indication': '{:+.4f}', 'indication_ci': '±{:.4f}',
                            'contraindication': '{:+.4f}', 'contraindication_ci': '±{:.4f}',
                            'micro': '{:+.4f}', 'micro_ci': '±{:.4f}'}, na_rep='-'))
    print('\n"favours_a" counts the seeds on which the first arm scored higher, out of n.')
else:
    print('Not enough arms finished for these contrasts yet.')

### 8.3 The degeneracy check

`dd_only` scores exactly 0.5000 ± 0.0000 because the disease signature is empty, so every disease is
scored by an identical prototype and the DistMult head cannot separate a true pair from a sampled
negative. If `dd_only_matched` — the same graph with an 8.3× larger encoder — also returns 0.5000,
capacity is irrelevant to that result and the only cause left is the missing signature.

In [ ]:
if wide is not None:
    present = wide.index.get_level_values('arm').unique()
    for a in ['dd_only', 'dd_only_matched']:
        if a not in present:
            print('%-18s no runs yet' % a)
            continue
        w = wide.xs(a, level='arm')
        p = int(long[long.arm == a].encoder_params.dropna().iloc[0])
        print('%-18s n=%2d | encoder %9s | indication %.4f ± %.4f | contra %.4f | micro %.4f'
              % (a, len(w), format(p, ','), w.indication.mean(), w.indication.std(ddof=0),
                 w.contraindication.mean(), w.micro.mean()))
    if 'dd_only_matched' in present:
        w = wide.xs('dd_only_matched', level='arm')
        at_chance = bool(np.allclose(w[['indication', 'contraindication', 'micro']].values, 0.5,
                                     atol=5e-4))
        print('\ndd_only_matched is at chance on every metric and seed: %s' % at_chance)
        print('-> capacity %s rescue an empty signature.' % ('does not' if at_chance else 'MAY'))

### 8.4 Save the summary

In [ ]:
if summary is not None:
    out_csv = ABL_ROOT / 'capacity_matched_graph_summary.csv'
    summary.to_csv(out_csv, index=False)
    print('wrote', out_csv.relative_to(REPO_ROOT))

    if wide is not None:
        paired = ABL_ROOT / 'capacity_matched_graph_per_seed.csv'
        wide.reset_index().to_csv(paired, index=False)
        print('wrote', paired.relative_to(REPO_ROOT))

    n_imported = int(long.groupby(['arm', 'seed']).imported.first().sum())
    n_total    = int(long.groupby(['arm', 'seed']).ngroups)
    print('\n%d run(s) on disk: %d trained here, %d imported from earlier grids.'
          % (n_total, n_total - n_imported, n_imported))

---
## 9 · What this grid can and cannot support

**Can.**

- Every keep-only loss can now be quoted at matched encoder capacity instead of as an upper bound,
  which is what a reviewer asking "did you just shrink the model?" needs.
- Whether relation *identity* still beats relation *count* once both arms have a full-size encoder.
- Whether the `dd_only` degeneracy is a capacity artifact. It is a prediction with one outcome that
  would falsify it, which is the point of running it.

**Cannot.**

- Matched *encoder* capacity is not matched *total* capacity: `n_out` widening also grows `w_rels` and
  the gate by a few thousand parameters. Reported, but too small to carry a result.
- `n_inp` stays at 100 on purpose, so the input features are equally impoverished in every arm. An arm
  with a wider input would also have richer frozen noise to work from, which is a different experiment.
- Widening is not the only way to restore capacity. Grid 10 showed the parameter-to-performance curve
  is non-monotonic, so "matched parameter count" is not the same as "matched effective capacity". The
  honest framing is that these arms remove the *parameter-count* objection, not every capacity
  objection.
- Nothing here speaks to the ranking protocol; these are edge-level AUPRCs with chance at 0.500.